In [1]:
import random
import pandas as pd

# 1. KITA BEKING BANK KATA (Bisa ditambahin lagi dari Kamus Manado)
subjek = ["Kita", "Ngana", "Dia", "Torang", "Dorang", "Ngoni", "Tu dosen", "Mama", "Paitua", "Maitua"]
predikat = ["so beking", "lupa bawa", "mo pigi cari", "ada lia", "so jual", "nyanda suka", "ba tele", "so makang"]
objek = ["tugas skripsi", "oto", "makanan", "doi", "baju baru", "hp", "laporan", "kunci rumah"]
keterangan = ["di kampus", "kemarin sore", "di Megamas", "cepat-cepat", "diam-diam", "pas ujang", "tadi pagi"]
akhiran = ["dang", "kang", "neh", "jo", "kote", "sto", "to"]

# Mapping sederhana ke Indo (buat label training)
map_subjek = {"Kita": "Saya", "Ngana": "Kamu", "Dia": "Dia", "Torang": "Kami", "Dorang": "Mereka", "Ngoni": "Kalian", "Tu dosen": "Dosen itu", "Mama": "Ibu", "Paitua": "Pacar (lk)", "Maitua": "Pacar (pr)"}
map_predikat = {"so beking": "sudah membuat", "lupa bawa": "lupa membawa", "mo pigi cari": "akan pergi mencari", "ada lia": "melihat", "so jual": "sudah menjual", "nyanda suka": "tidak suka", "ba tele": "menelepon", "so makang": "sudah makan"}
map_objek = {"tugas skripsi": "tugas skripsi", "oto": "mobil", "makanan": "makanan", "doi": "uang", "baju baru": "baju baru", "hp": "handphone", "laporan": "laporan", "kunci rumah": "kunci rumah"}
map_keterangan = {"di kampus": "di kampus", "kemarin sore": "kemarin sore", "di Megamas": "di Megamas", "cepat-cepat": "dengan cepat", "diam-diam": "diam-diam", "pas ujang": "saat hujan", "tadi pagi": "tadi pagi"}

dataset = []

# 2. GENERATE 1000 KALIMAT
for i in range(1000):
    s = random.choice(subjek)
    p = random.choice(predikat)
    o = random.choice(objek)
    k = random.choice(keterangan)
    a = random.choice(akhiran)

    # Kalimat Manado
    kalimat_manado = f"{s} {p} {o} {k} {a}."

    # Kalimat Indo (Terjemahan Kasar)
    kalimat_indo = f"{map_subjek[s]} {map_predikat[p]} {map_objek[o]} {map_keterangan[k]}."

    dataset.append([kalimat_manado, kalimat_indo])

# 3. SIMPAN
df_sintetis = pd.DataFrame(dataset, columns=['Teks_Manado', 'Teks_Indo'])
print("🎉 Mantap! So jadi 1000 data.")
print(df_sintetis.head())

# Download
df_sintetis.to_csv('dataset_manado_sintetis_1000.csv', index=False)

🎉 Mantap! So jadi 1000 data.
                                        Teks_Manado  \
0     Torang so beking kunci rumah cepat-cepat sto.   
1             Dia so makang makanan pas ujang dang.   
2       Ngoni so jual tugas skripsi di Megamas sto.   
3              Dia mo pigi cari oto cepat-cepat jo.   
4  Dorang mo pigi cari baju baru kemarin sore dang.   

                                           Teks_Indo  
0       Kami sudah membuat kunci rumah dengan cepat.  
1                Dia sudah makan makanan saat hujan.  
2     Kalian sudah menjual tugas skripsi di Megamas.  
3         Dia akan pergi mencari mobil dengan cepat.  
4  Mereka akan pergi mencari baju baru kemarin sore.  


In [2]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split

# --- 1. FUNGSI MEMBERSIHKAN TEKS (CLEANING) ---
def bersihkan_teks(teks):
    if not isinstance(teks, str): return ""

    # Ganti huruf kecil semua (Case Folding) - Bagus buat BERT awal
    teks = teks.lower()

    # Hapus URL (http/https/www)
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)

    # Hapus Mention (@username) dan Hashtag (#)
    teks = re.sub(r'@\w+|#\w+', '', teks)

    # Hapus Emoji dan Simbol Aneh (Simpan titik/koma/tanda tanya/seru)
    teks = re.sub(r'[^a-z0-9\s.,!?]', '', teks)

    # Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()

    return teks

print("🚀 MEMULAI PROSES PENGGABUNGAN DATA...")

# --- 2. LOAD DATASET (Pastikan nama file sesuai yg tadi torang bikin) ---
data_gabungan = []

# A. Data Facebook
try:
    df_fb = pd.read_csv('dataset_ig_manado.csv')
    # Sesuaikan nama kolom dg yg di CSV FB
    col_name = 'Teks_Manado_FB' if 'Teks_Manado_FB' in df_fb.columns else df_fb.columns[0]
    print(f"✅ Load FB: {len(df_fb)} baris")
    data_gabungan.extend(df_fb[col_name].tolist())
except Exception as e:
    print(f"⚠️ Data IG belum ada/error: {e}")

# B. Data YouTube
try:
    df_yt = pd.read_csv('dataset_youtube_jadi.csv')
    col_name = 'Komentar_Manado' if 'Komentar_Manado' in df_yt.columns else df_yt.columns[0]
    print(f"✅ Load YouTube: {len(df_yt)} baris")
    data_gabungan.extend(df_yt[col_name].tolist())
except Exception as e:
    print(f"⚠️ Data YouTube belum ada/error: {e}")

# C. Data Sintetis (Pabrik Kalimat)
try:
    df_syn = pd.read_csv('dataset_manado_sintetis_1000.csv')
    col_name = 'Teks_Manado' if 'Teks_Manado' in df_syn.columns else df_syn.columns[0]
    print(f"✅ Load Sintetis: {len(df_syn)} baris")
    data_gabungan.extend(df_syn[col_name].tolist())
except Exception as e:
    print(f"⚠️ Data Sintetis belum ada/error: {e}")

# --- 3. PROSES CLEANING & FILTERING ---
print("\n🧹 Sedang membersihkan data...")

df_final = pd.DataFrame(data_gabungan, columns=['teks_raw'])

# Terapkan fungsi cleaning
df_final['teks_bersih'] = df_final['teks_raw'].apply(bersihkan_teks)

# Hapus data kosong atau terlalu pendek (kurang dari 3 huruf)
df_final = df_final[df_final['teks_bersih'].str.len() > 3]

# HAPUS DUPLIKAT (Penting biar training efisien)
jumlah_awal = len(df_final)
df_final = df_final.drop_duplicates(subset=['teks_bersih'])
jumlah_akhir = len(df_final)

print(f"   Dibuang {jumlah_awal - jumlah_akhir} data duplikat/sampah.")
print(f"🎉 TOTAL DATASET BERSIH: {jumlah_akhir} Kalimat.")

# --- 4. SPLITTING (TRAIN vs VALIDATION) ---
# Bagi 85% Train, 15% Test
train, val = train_test_split(df_final['teks_bersih'], test_size=0.15, random_state=42)

print(f"\n📊 Statistik:")
print(f"   - Data Training: {len(train)} baris")
print(f"   - Data Validasi: {len(val)} baris")

# --- 5. SIMPAN KE TXT (Format Dataset BERT) ---
# Kita simpan per baris (line-by-line)
train.to_csv('data_train_manado.txt', index=False, header=False)
val.to_csv('data_val_manado.txt', index=False, header=False)

print("\n💾 SUKSES! File 'data_train_manado.txt' dan 'data_val_manado.txt' siap dipake!")
print("👉 Contoh 5 Kalimat Bersih:")
print(train.head().values)

🚀 MEMULAI PROSES PENGGABUNGAN DATA...
✅ Load FB: 20 baris
✅ Load YouTube: 90 baris
✅ Load Sintetis: 1000 baris

🧹 Sedang membersihkan data...
   Dibuang 13 data duplikat/sampah.
🎉 TOTAL DATASET BERSIH: 1097 Kalimat.

📊 Statistik:
   - Data Training: 932 baris
   - Data Validasi: 165 baris

💾 SUKSES! File 'data_train_manado.txt' dan 'data_val_manado.txt' siap dipake!
👉 Contoh 5 Kalimat Bersih:
['kita so jual tugas skripsi di megamas jo.'
 'dia so jual oto cepatcepat to.' 'paitua so jual oto di kampus to.'
 'maitua mo pigi cari makanan tadi pagi to.'
 'ngoni so makang doi di kampus neh.']
